In [1]:
import json
import pandas as pd
from pathlib import Path


In [17]:
div_folder = Path('../../data/this_project/5_div/')
mapping_fn = div_folder / '5A_metabolomics_name_to_bigg.csv'
mapping_df = pd.read_csv(mapping_fn)

Z_div_folder = Path('../../data/this_project/6_transporterKO/Z_div')
ecmdb_kegg_fn = Z_div_folder / 'C_ecmdb_kegg_ids.csv'

In [20]:
import json

# Load ECMDB JSON and build useful lookup dictionaries
with open('../../data/other/ecmdb.json') as f:
    ecmdb_data = json.load(f)

# ECMDB metabolite ID -> KEGG ID (skip entries missing either field)
ecmdb_id_to_kegg_id = {
    entry['met_id']: entry['kegg_id']
    for entry in ecmdb_data
    if entry.get('met_id') and entry.get('kegg_id')
}


In [ ]:
mapping_df

,Metabolite,BiGG ID,ECMDB ID,ECMDB name,In iML1515,Ecocyc
0,"1,3-diaminopropane",13dampp,NaN,NaN,False,NaN
1,"2,5-dihydroxybenzoate",23dhb,NaN,NaN,True,NaN
2,2-aminoisobutyrate,NaN,ECMDB21403,2-Aminoisobutyric acid,NaN,0.0
3,2-hydroxybutyrate,ghb,ECMDB24007,2-Hydroxybutyric acid,True,NaN
4,2-hydroxyglutaric acid,S2hglut,ECMDB00606,D-2-Hydroxyglutaric acid,True,NaN
...,...,...,...,...,...,...
123,Urocanate,urcan,ECMDB21396,Urocanic acid,False,NaN
124,Valine,val__L,ECMDB00883,L-Valine,True,NaN
125,Xanthine,xan,ECMDB00292,Xanthine,True,NaN
126,Xanthosine,xtsn,ECMDB00299,Xanthosine,True,NaN


In [ ]:
mapping_df['KEGG_ID'] = mapping_df['ECMDB ID'].map(ecmdb_id_to_kegg_id)

In [ ]:
mapping_df.loc[mapping_df['KEGG_ID'].isna(), ['Metabolite', 'BiGG ID', 'ECMDB ID', 'KEGG_ID']].to_csv('temp.csv', index=False)

# Manual curation of the remaining ones that maps

In [ ]:
# KEGG IDs found for metabolites that didn't match in the original lookup.
# Keys match the 'Metabolite' strings exactly as they appear in your CSV.
kegg_id_map = {
    "1,3-diaminopropane": "C00986",
    "2,5-dihydroxybenzoate": "C00628",       
    "2-hydroxybutyrate": "C05984",
    "4-imidazoleacetate": "C02835",
    "Adipate": "C06104",
    "Alpha-aminobutyrate": "C02356",         # NOTE: this is specifically S-2-aminobutyrate
    "Cdp-ethanolamine": "C00570",
    "Creatine": "C00300",
    "Creatinine": "C00791",
    "Hexoses": "C00124",                     # mapped as D-galactose as that is used as C-source
    "Kynurenate": "C01717",
    "Metanephrine": "C05588",
    "Methylguanidine": "C02294",
    "N-acetylleucine": "C02710",
    "Oxoproline": "C01879",                  # 5-Oxo-L-proline (glutathione-pathway form); KEGG also has C02237 for the D-form
    "P-hydroxyphenylacetate": "C00642",
    "Trigonelline ": "C01004",               
}

mapping_df["KEGG_ID"] = mapping_df["KEGG_ID"].fillna(mapping_df["Metabolite"].map(kegg_id_map))

# Map to ecmdb metabolites


In [24]:
ecmdb_kegg_ids = pd.read_csv(ecmdb_kegg_fn, header=None, names=['KEGG_ID'])['KEGG_ID'].tolist()

In [ ]:
mapping_df['IN ECMDB'] = False

for i, row in mapping_df.iterrows():
    ecmdb_id = row['ECMDB ID']
    kegg_id = row['KEGG_ID']

    has_ecmdb_id = isinstance(ecmdb_id, str) and ecmdb_id.startswith('ECMDB')
    has_matching_kegg = isinstance(kegg_id, str) and kegg_id.strip() in ecmdb_kegg_ids

    mapping_df.at[i, 'IN ECMDB'] = has_ecmdb_id or has_matching_kegg


In [ ]:
ecmdb_kegg_ids

['C00109',
 'C00526',
 'C00881',
 'C00141',
 'C01188',
 'C02642',
 'C00120',
 'C00147',
 'C00246',
 'C00033',
 'C00072',
 'C00020',
 'C05402',
 'C00212',
 'C00014',
 'C00185',
 'C00099',
 'C00575',
 'C00164',
 'C00054',
 'C00487',
 'C05512',
 'C00417',
 'C03758',
 'C00429',
 'C01419',
 'C00906',
 'C00063',
 'C00330',
 'C00670',
 'C00475',
 'C00158',
 'C00055',
 'C00114',
 'C00181',
 'C02291',
 'C00559',
 'C01697',
 'C00469',
 'C01233',
 'C00048',
 'C00504',
 'C00031',
 'C00037',
 'C00085',
 'C00051',
 'C00093',
 'C00116',
 'C00242',
 'C00387',
 'C00058',
 'C00124',
 'C00025',
 'C00189',
 'C00198',
 'C00149',
 'C00262',
 'C00082',
 'C00079',
 'C00137',
 'C00041',
 'C00148',
 'C00208',
 'C00155',
 'C00188',
 'C00152',
 'C00159',
 'C00407',
 'C01019',
 'C00122',
 'C00135',
 'C00047',
 'C00243',
 'C00065',
 'C00081',
 'C00186',
 'C00049',
 'C00491',
 'C00294',
 'C00166',
 'C00864',
 'C04006',
 'C01602',
 'C00140',
 'C00006',
 'C01103',
 'C00249',
 'C00036',
 'C00322',
 'C00295',
 'C01185',

In [ ]:
mapping_df.to_csv(mapping_fn, index=False)

# Map KEIO KO metabolomics

In [27]:
data_folder_KEIO_KO = Path('../../data/this_project/2_keio_strains_screening/')

mapping_df2 = pd.read_csv(data_folder_KEIO_KO / '2F_metabolomics_name_to_bigg.csv')


In [28]:
new_mapping_df2 = []

for _, row in mapping_df2.iterrows():
    metabolite = str(row['Metabolite']).strip()
    matched_rows = mapping_df[mapping_df['Metabolite'].astype(str).str.strip() == metabolite]

    if not matched_rows.empty:
        matched_row = matched_rows.iloc[0]
        new_mapping_df2.append({
            'Metabolite': metabolite,
            'BiGG ID': row['BiGG ID'],
            'ECMDB ID': matched_row['ECMDB ID'],
            'KEGG_ID': matched_row['KEGG_ID'],
            'IN ECMDB': matched_row['IN ECMDB'],
            'From Mapping 1': True
        })
    else:
        new_mapping_df2.append({
            'Metabolite': metabolite,
            'BiGG ID': row['BiGG ID'],
            'ECMDB ID': None,
            'KEGG_ID': None,
            'IN ECMDB': False,
            'From Mapping 1': False
        })

new_mapping_df2 = pd.DataFrame(new_mapping_df2)


In [29]:
mapping_df.loc[mapping_df.Metabolite == 'Carnitine (c0)']

,Metabolite,BiGG ID,ECMDB ID,ECMDB name,In iML1515,Ecocyc,KEGG_ID,IN ECMDB


In [31]:
# KEGG IDs extracted directly from iML1515.xml (BiGG model annotations).
# Keys match the 'Metabolite' strings exactly as they appear in your CSV.
kegg_id_map2 = {
    "Adenosine 3', 5'-cyclic phosphate": "C00575",
    "2-oxobutanoate/acetoacetate": "C00109",       # model: 2-Oxobutanoate (not acetoacetate)
    "Biotin": "C00120",
    "Cadaverine": "C01672",
    "Carnitine (c0)": "C00318",                     # model also cross-refs C00487
    "Choline": "C00114",
    "FAD": "C00016",
    "Glucosamine 6-phosphate": "C00352",
    "Glutathione reduced": "C00051",
    "Histidinol": "C00860",
    "Mannose": "C00159",
    "Pyridoxal": "C00250",
    "Riboflavin": "C00255",
    "Ribose 5-phosphate": "C03736",                 # alpha-anomer specific; C00117 is the generic form
    "Sedoheptulose-7-phopsphate": "C05382",         # matches the typo in your original file
    "Uridine diphosphate glucuronic acid": "C00167",
    "2,5-dihydroxybenzoate": "C00628"
}

kegg_id_to_ecmdb_id = {v: k for k, v in ecmdb_id_to_kegg_id.items()}
ecmdb_id_map2 = {}
for name, kegg_id in kegg_id_map2.items():
    ecmdb_id = kegg_id_to_ecmdb_id.get(kegg_id)
    if ecmdb_id:
        ecmdb_id_map2[name] = ecmdb_id

ecmdb_id_map2["Adenosine 3', 5'-cyclic phosphate"] = "ECMDB00058"
ecmdb_id_map2["Ribose 5-phosphate"] = "ECMDB02033"

In [32]:
for _, row in new_mapping_df2.iterrows():
    metabolite = str(row['Metabolite']).strip()
    if metabolite in kegg_id_map2:
        new_mapping_df2.at[_, 'KEGG_ID'] = kegg_id_map2[metabolite]
        new_mapping_df2.at[_, 'ECMDB ID'] = ecmdb_id_map2.get(metabolite)
    

for i, row in new_mapping_df2.iterrows():
    ecmdb_id = row['ECMDB ID']
    kegg_id = row['KEGG_ID']
    has_ecmdb_id = isinstance(ecmdb_id, str) and ecmdb_id.startswith('ECMDB')
    has_matching_kegg = isinstance(kegg_id, str) and kegg_id.strip() in ecmdb_kegg_ids
    if has_ecmdb_id or has_matching_kegg:
        new_mapping_df2.at[i, 'IN ECMDB'] = True


In [ ]:
mapping_df2.to_csv(data_folder_KEIO_KO / '2F_metabolomics_name_to_bigg.csv')


,Metabolite,BiGG ID,ECMDB ID,KEGG_ID,IN ECMDB,From Mapping 1
0,"1,3-diaminopropane",13dampp,NaN,C00986,False,True
1,"2, 5-dihydroxybenzoate",NaN,None,None,False,False
2,"2,5-dihydroxybenzoate",NaN,None,C00628,False,True
3,2-aminoisobutyrate,NaN,ECMDB21403,C03665,True,True
4,2-hydroxybutyrate,ghb,ECMDB24007,C05984,True,True
...,...,...,...,...,...,...
139,Urocanate,urcan,ECMDB21396,C00785,True,True
140,Valine,val__L,ECMDB00883,C00183,True,True
141,Xanthine,xan,ECMDB00292,C00385,True,True
142,Xanthosine,xtsn,ECMDB00299,C01762,True,True
